# PostgreSQL 18 — Realistic Seed with 80-Question Coverage

**Destructive reset:** recreates only the 15 interview tables in `public` after typing `RESET`. It does not modify the `uber_interview` schema.

Designed for the original 80 SQL questions. Includes guaranteed never-ordered products, no-order customers, repeated purchases, consecutive logins, staging duplicates, payment duplicates, SLA breaches, reconciliation mismatches, stock shortages, and suspicious transactions.

**Run the separate validation notebook after seeding.** Live PostgreSQL execution and actual row counts are verified on your machine, not pre-certified here.


In [ ]:
import random, datetime as dt, decimal
import psycopg2
from psycopg2.extras import execute_values
from IPython.display import display
import pandas as pd

# Import the shared database configuration regardless of whether VS Code
# starts the kernel in coding-exercises/ or coding-exercises/sql/.
from pathlib import Path
import sys
_SQL_DIR = next((p for p in (Path.cwd(), Path.cwd() / "sql")
                 if (p / "db_connection.py").is_file()), None)
if _SQL_DIR is None:
    raise FileNotFoundError(
        "Cannot locate sql/db_connection.py. Open coding-exercises/ or its sql/ folder in VS Code."
    )
if str(_SQL_DIR) not in sys.path:
    sys.path.insert(0, str(_SQL_DIR))
from db_connection import connect, connection_settings

conn = connect(autocommit=False)
rng = random.Random(20261009)
print('Connected:', connection_settings()['dbname'], '| current schema: public')


## 1. Explicit reset and table recreation
Only the 15 practice tables in `public` are affected. All SQL interview notebooks can continue using unqualified table names.

In [ ]:
TABLES = ['departments','employees','customers','products','orders','order_items',
          'support_tickets','projects','project_assignments','payments','events',
          'transactions','customer_staging','product_staging','pipeline_logs']

DDL = """
CREATE TABLE public.departments (department_id INT PRIMARY KEY, department_name VARCHAR(60) NOT NULL, location VARCHAR(60) NOT NULL);
CREATE TABLE public.employees (employee_id INT PRIMARY KEY, first_name VARCHAR(40), last_name VARCHAR(40), email VARCHAR(100) UNIQUE, department_id INT REFERENCES public.departments, manager_id INT REFERENCES public.employees(employee_id), hire_date DATE, salary NUMERIC(12,2), job_title VARCHAR(80));
CREATE TABLE public.customers (customer_id INT PRIMARY KEY, first_name VARCHAR(40), last_name VARCHAR(40), email VARCHAR(100) UNIQUE, city VARCHAR(60), province VARCHAR(40), segment VARCHAR(30), signup_date DATE);
CREATE TABLE public.products (product_id INT PRIMARY KEY, sku VARCHAR(30) UNIQUE, product_name VARCHAR(100), category VARCHAR(50), unit_price NUMERIC(10,2), stock_quantity INT, reorder_level INT, active BOOLEAN);
CREATE TABLE public.orders (order_id INT PRIMARY KEY, customer_id INT REFERENCES public.customers, sales_employee_id INT REFERENCES public.employees, order_date DATE, status VARCHAR(20), total_amount NUMERIC(12,2));
CREATE TABLE public.order_items (order_item_id INT PRIMARY KEY, order_id INT REFERENCES public.orders, product_id INT REFERENCES public.products, quantity INT, unit_price NUMERIC(10,2));
CREATE TABLE public.support_tickets (ticket_id INT PRIMARY KEY, customer_id INT REFERENCES public.customers, assigned_employee_id INT REFERENCES public.employees, created_at TIMESTAMP, resolved_at TIMESTAMP, priority VARCHAR(15), status VARCHAR(20), subject VARCHAR(120));
CREATE TABLE public.projects (project_id INT PRIMARY KEY, project_name VARCHAR(100), start_date DATE, end_date DATE, status VARCHAR(20));
CREATE TABLE public.project_assignments (assignment_id INT PRIMARY KEY, project_id INT REFERENCES public.projects, employee_id INT REFERENCES public.employees, role_on_project VARCHAR(60), allocation_pct INT);
CREATE TABLE public.payments (payment_id INT PRIMARY KEY, order_id INT REFERENCES public.orders, payment_date TIMESTAMP, amount NUMERIC(12,2), payment_method VARCHAR(20), payment_status VARCHAR(20));
CREATE TABLE public.events (event_id INT PRIMARY KEY, user_id INT, event_type VARCHAR(30), event_ts TIMESTAMP, session_id VARCHAR(40), page VARCHAR(80));
CREATE TABLE public.transactions (transaction_id INT PRIMARY KEY, customer_id INT REFERENCES public.customers, transaction_ts TIMESTAMP, transaction_type VARCHAR(20), amount NUMERIC(12,2), channel VARCHAR(20));
CREATE TABLE public.customer_staging (staging_id INT PRIMARY KEY, source_customer_id INT, first_name VARCHAR(40), last_name VARCHAR(40), email VARCHAR(100), city VARCHAR(60), updated_at TIMESTAMP);
CREATE TABLE public.product_staging (staging_id INT PRIMARY KEY, sku VARCHAR(30), product_name VARCHAR(100), category VARCHAR(50), unit_price NUMERIC(10,2), updated_at TIMESTAMP);
CREATE TABLE public.pipeline_logs (log_id INT PRIMARY KEY, pipeline_name VARCHAR(80), log_level VARCHAR(10), message VARCHAR(200), logged_at TIMESTAMP);
"""

confirmation = input('Type RESET to drop/recreate the 15 public practice tables: ')
if confirmation != 'RESET':
    raise RuntimeError('Cancelled: no tables changed.')
try:
    with conn.cursor() as cur:
        # DROP CASCADE is not used: external dependencies will prevent accidental deletion.
        for table in reversed(TABLES):
            cur.execute(f'DROP TABLE IF EXISTS public.{table}')
        cur.execute(DDL)
    conn.commit()
    print('15 tables recreated in public.')
except Exception:
    conn.rollback()
    raise

## 2. Generate coherent, realistic seed data

In [ ]:
FIRST = ['Olivia','Liam','Emma','Noah','Ava','Ethan','Sophia','Lucas','Mia','Benjamin','Charlotte','Oliver','Amelia','Elijah','Isabella','James','Harper','William','Evelyn','Henry','Abigail','Alexander','Emily','Daniel','Ella','Matthew','Grace','Michael','Chloe','Samuel','Lily','David','Aria','Joseph','Zoe','Andrew','Nora','Nathan','Hannah','Ryan','Leah','Adam','Sofia','Sarah','Jack','Victoria','Maya','Logan','Layla','Leo','Claire','Isaac','Ruby','Gabriel','Anna','Owen','Alice','Caleb','Stella','Thomas','Nina','Jacob','Elena','Aiden','Julia','Dylan','Lucy','Anthony','Eva','Christopher','Audrey','Sebastian','Ivy','Joshua','Bella','Adrian','Naomi','Aaron','Mila','Julian','Ellie','Connor','Jasmine','Isaiah','Madison','Felix','Sienna','Patrick','Freya','Eric','Isla','Jason','Rose','Peter','Amara','Simon','Aisha','Amir','Priya','Fatima','Mei','Arjun','Hiro','Mateo','Sana','Nadia','Yusuf','Omar','Amina','Leila','Celine','Elias','Anika','Marco','Bianca','Hugo','Ines','Ravi','Tara','Noor']
LAST = ['Smith','Johnson','Brown','Wilson','Taylor','Anderson','Thomas','Moore','Martin','Lee','Thompson','White','Harris','Clark','Lewis','Walker','Hall','Young','Allen','King','Wright','Scott','Green','Baker','Adams','Nelson','Carter','Mitchell','Perez','Roberts','Turner','Phillips','Campbell','Parker','Evans','Edwards','Collins','Stewart','Morris','Rogers','Reed','Cook','Morgan','Bell','Murphy','Bailey','Rivera','Cooper','Richardson','Cox','Howard','Ward','Torres','Peterson','Gray','Ramirez','James','Watson','Brooks','Kelly','Sanders','Price','Bennett','Wood','Barnes','Ross','Henderson','Coleman','Jenkins','Perry','Powell','Long','Patterson','Hughes','Flores','Washington','Butler','Simmons','Foster','Gonzalez','Bryant','Alexander','Russell','Griffin','Diaz','Hayes','Myers','Ford','Hamilton','Graham','Sullivan','Wallace','Woods','West','Jordan','Owens','Reynolds','Fisher','Ellis','Harrison','Gibson','McDonald','Cruz','Marshall','Ortiz','Gomez','Murray','Freeman','Wells','Webb','Simpson','Stevens','Tucker','Porter','Hunter','Hicks','Crawford','Henry','Boyd','Mason']
PLACES = [('Vancouver','BC'),('Burnaby','BC'),('Richmond','BC'),('Surrey','BC'),('Langley','BC'),('Coquitlam','BC'),('Victoria','BC'),('Kelowna','BC'),('Toronto','ON'),('Ottawa','ON'),('Mississauga','ON'),('Waterloo','ON'),('Calgary','AB'),('Edmonton','AB'),('Montreal','QC'),('Quebec City','QC'),('Halifax','NS'),('Winnipeg','MB')]
DEPTS = ['Data Engineering','Analytics','Finance','Sales','Customer Support','Cloud Platform','Information Security','Product','Operations','Marketing']
CATALOG = [
('Wireless Noise-Cancelling Headphones','Audio',179.99),('Bluetooth Earbuds','Audio',89.95),('USB Microphone','Audio',74.50),('Portable Bluetooth Speaker','Audio',119.00),
('Mechanical Keyboard','Accessories',129.99),('Wireless Keyboard','Accessories',59.99),('Ergonomic Mouse','Accessories',69.00),('USB-C Hub','Accessories',44.95),('Laptop Stand','Office',49.95),('Ergonomic Desk Chair','Office',349.00),
('27-inch 4K Monitor','Displays',399.99),('32-inch QHD Monitor','Displays',329.00),('Portable 15-inch Monitor','Displays',249.00),('Webcam 1080p','Video',79.99),('4K Webcam','Video',149.00),
('Portable SSD 1TB','Storage',119.99),('Portable SSD 2TB','Storage',199.99),('USB Flash Drive 128GB','Storage',24.99),('NAS Drive 4TB','Storage',219.00),
('Wi-Fi 6 Router','Networking',139.99),('Mesh Wi-Fi System','Networking',289.00),('Gigabit Ethernet Switch','Networking',79.00),('USB-C Docking Station','Accessories',189.99),('65W USB-C Charger','Accessories',39.99),
('Smart LED Light Strip','Smart Home',39.00),('Smart Plug 2-Pack','Smart Home',29.99),('Video Doorbell','Smart Home',159.99),('Smart Thermostat','Smart Home',229.00),
('Laptop Backpack','Office',79.00),('Desk Organizer','Office',28.00),('External Hard Drive 2TB','Storage',99.99),('Monitor Arm','Office',99.00),
('Noise-Cancelling Headset','Audio',139.00),('HDMI Cable 2m','Accessories',14.99),('USB-C Cable 1m','Accessories',12.99),('Wireless Presenter','Office',39.00),
('Network Adapter','Networking',29.99),('4K Streaming Camera','Video',189.99),('Screen Light Bar','Office',69.00),('Surge Protector','Office',34.99)]
BRANDS=['Logitech','Anker','Samsung','Dell','Belkin','Sony','JBL','Lenovo']
BASE = dt.date(2024, 1, 1)
def day(start, end): return start + dt.timedelta(days=rng.randint(0,(end-start).days))
def money(x): return round(float(x),2)
def batch(table, columns, rows):
    with conn.cursor() as cur:
        execute_values(cur, f'INSERT INTO public.{table} ({", ".join(columns)}) VALUES %s', rows, page_size=500)
    print(f'{table:22s} {len(rows):>5} rows')

try:
    departments=[(i+1,DEPTS[i%10],PLACES[(i//10*3+i%10)%len(PLACES)][0]) for i in range(50)]
    batch('departments',['department_id','department_name','location'],departments)

    employee_names=rng.sample([(f,l) for f in FIRST for l in LAST],100)
    employees=[]
    for i,(first,last) in enumerate(employee_names,1):
        manager=None if i<=10 else rng.randint(1,10)
        job='Engineering Manager' if i<=10 else rng.choice(['Data Engineer','Analytics Engineer','Business Analyst','Software Engineer','Support Specialist','Product Analyst','Cloud Engineer','Data Analyst'])
        salary=rng.randrange(120000,170000,2500) if i<=10 else rng.randrange(68000,145000,1500)
        employees.append((i,first,last,f'{first.lower()}.{last.lower()}.{i}@northstar.example',((i-1)%50)+1,manager,day(dt.date(2017,1,1),dt.date(2025,9,30)),salary,job))
    batch('employees',['employee_id','first_name','last_name','email','department_id','manager_id','hire_date','salary','job_title'],employees)

    customer_names=rng.sample([(f,l) for f in FIRST for l in LAST if (f,l) not in employee_names],120)
    customers=[]
    for i,(first,last) in enumerate(customer_names,1):
        city,province=rng.choice(PLACES)
        customers.append((i,first,last,f'{first.lower()}.{last.lower()}.{i}@example.com',city,province,rng.choices(['Consumer','SMB','Enterprise'],[70,23,7])[0],day(dt.date(2023,1,1),dt.date(2026,7,1))))
    batch('customers',['customer_id','first_name','last_name','email','city','province','segment','signup_date'],customers)

    products=[]
    for i in range(1,81):
        name,category,base=CATALOG[(i-1)%len(CATALOG)]
        brand=BRANDS[((i-1)//len(CATALOG)+i*3)%len(BRANDS)]
        price=money(base*(1+0.07*((i-1)//len(CATALOG))))
        products.append((i,f'SKU-{i:04d}',f'{brand} {name}',category,price,rng.randint(0,125),rng.randint(8,24),i%19!=0))
    # Q05: product IDs 76-80 are intentionally never ordered.
    # Q79: active, low-stock products for reorder analysis.
    products = [(*p[:5], 2, 15, True) if p[0] in (5, 15, 25) else p for p in products]
    batch('products',['product_id','sku','product_name','category','unit_price','stock_quantity','reorder_level','active'],products)

    orders=[]; items=[]; item_id=1
    for i in range(1,301):
        customer_id=rng.randint(1,110)
        order_date=day(dt.date(2025,1,1),dt.date(2026,9,30))
        status=rng.choices(['Completed','Shipped','Pending','Cancelled'],[68,14,12,6])[0]
        order_total=0
        for pid in rng.sample(range(1,76),rng.randint(1,5)):
            qty=rng.randint(1,3)
            price=products[pid-1][4]
            items.append((item_id,i,pid,qty,price)); item_id+=1
            order_total+=qty*price
        orders.append((i,customer_id,rng.randint(1,100),order_date,status,money(order_total)))
    # Controlled reconciliation mismatches for Q74
    orders=[(*o[:5],money(o[5]+4.99) if o[0] in (37,119,241) else o[5]) for o in orders]
    # Q75: guaranteed repeat customer inactive for >90 days.
    orders = [(*o[:1], 109, o[2], dt.date(2025,2,10) if o[0]==1 else dt.date(2025,3,10), *o[4:]) if o[0] in (1,2) else o for o in orders]
    # Q69: customer 109 has at least three orders, with one in a later month.
    orders = [(*o[:1], 109, o[2], dt.date(2025,4,10), *o[4:]) if o[0]==3 else o for o in orders]
    # Avoid any other orders for 109 so Q75 stays deterministic.
    orders = [(*o[:1], 108 if o[1]==109 and o[0]>3 else o[1], *o[2:]) for o in orders]
    batch('orders',['order_id','customer_id','sales_employee_id','order_date','status','total_amount'],orders)
    batch('order_items',['order_item_id','order_id','product_id','quantity','unit_price'],items)

    subjects=['Cannot sign in to account','Incorrect invoice amount','Order delivery delayed','Refund request','Product arrived damaged','Unable to download invoice','Payment method declined','Account verification issue','Missing order confirmation','Subscription cancellation request']
    tickets=[]
    for i in range(1,101):
        created=dt.datetime.combine(day(dt.date(2026,1,1),dt.date(2026,9,30)),dt.time(rng.randint(8,20),rng.choice([0,15,30,45])))
        status=rng.choices(['Resolved','Closed','Open'],[60,20,20])[0]
        resolved=created+dt.timedelta(hours=rng.randint(2,144)) if status!='Open' else None
        tickets.append((i,rng.randint(1,120),None if i%9==0 else rng.randint(1,100),created,resolved,rng.choice(['High','Medium','Low']),status,rng.choice(subjects)))
    batch('support_tickets',['ticket_id','customer_id','assigned_employee_id','created_at','resolved_at','priority','status','subject'],tickets)

    project_titles=['Customer 360 Platform','Cloud Data Lake Migration','Payments Reconciliation','Product Analytics Dashboard','Inventory Forecasting','Event Streaming Platform','Revenue Reporting Modernization','Data Quality Framework','Customer Churn Analytics','Warehouse Cost Optimization','CRM Data Integration','Fraud Detection Pipeline','Marketing Attribution','Observability and Alerting','Finance Data Mart','Identity Resolution','Self-Service Analytics','Order Fulfillment Insights','Privacy Compliance Automation','Recommendation Engine']
    projects=[]
    for i in range(1,51):
        start=day(dt.date(2024,1,1),dt.date(2026,4,1));end=start+dt.timedelta(days=rng.randint(90,420))
        status='Completed' if end<dt.date(2026,10,1) else ('Planned' if start>dt.date(2026,10,1) else 'Active')
        projects.append((i,project_titles[(i-1)%len(project_titles)]+(f' — Phase {1+(i-1)//len(project_titles)}' if i>len(project_titles) else ''),start,end,status))
    batch('projects',['project_id','project_name','start_date','end_date','status'],projects)
    assignments=[]
    for i in range(1,251):
        assignments.append((i,((i-1)%50)+1,rng.randint(1,100),rng.choice(['Engineer','Analyst','Lead','QA','Architect']),rng.choice([25,50,75,100])))
    batch('project_assignments',['assignment_id','project_id','employee_id','role_on_project','allocation_pct'],assignments)

    payments=[]
    for i,o in enumerate(orders,1):
        status=o[4]
        pstatus='Failed' if status=='Cancelled' else ('Pending' if status=='Pending' else 'Paid')
        payments.append((i,i,dt.datetime.combine(o[3],dt.time(12,0))+dt.timedelta(hours=rng.randint(0,72)),o[5],rng.choice(['Card','PayPal','Bank Transfer']),pstatus))
    for i in range(1,21):
        o=orders[i*10-1]
        payments.append((300+i,o[0],dt.datetime.combine(o[3],dt.time(12,0))+dt.timedelta(days=2),o[5],'Card','Failed' if i%2 else 'Paid'))
    batch('payments',['payment_id','order_id','payment_date','amount','payment_method','payment_status'],payments)

    event_types=[('login','/login'),('product_view','/products'),('product_view','/product/detail'),('add_to_cart','/cart'),('purchase','/checkout')]
    events=[]
    for i in range(1,701):
        etype,page=rng.choice(event_types)
        ts=dt.datetime.combine(day(dt.date(2026,6,1),dt.date(2026,9,30)),dt.time(rng.randint(0,23),rng.randint(0,59)))
        events.append((i,rng.randint(1,120),etype,ts,f'S-{rng.randint(1,180):04d}',page))
    for i in range(1,21): events.append((700+i,*events[i-1][1:]))
    # Q46: two consecutive login dates for the same user, plus a third date.
    events += [
        (721,1,'login',dt.datetime(2026,9,21,8,15),'S-LOGIN-1','/login'),
        (722,1,'login',dt.datetime(2026,9,22,9,10),'S-LOGIN-2','/login'),
        (723,1,'login',dt.datetime(2026,9,23,10,5),'S-LOGIN-3','/login'),
    ]
    batch('events',['event_id','user_id','event_type','event_ts','session_id','page'],events)

    transactions=[]
    for i in range(1,501):
        t=dt.datetime.combine(day(dt.date(2026,1,1),dt.date(2026,9,30)),dt.time(rng.randint(0,23),rng.randint(0,59)))
        transactions.append((i,rng.randint(1,120),t,rng.choices(['Purchase','Refund','Subscription'],[75,12,13])[0],money(rng.uniform(20,1100)),rng.choice(['Web','Mobile','Store'])))
    for i in range(1,31):
        customer=1+(i-1)//3
        transactions.append((500+i,customer,dt.datetime(2026,9,20,10)+dt.timedelta(days=customer-1,hours=i%3),'Purchase',450+(i%3)*25,'Web'))
    batch('transactions',['transaction_id','customer_id','transaction_ts','transaction_type','amount','channel'],transactions)

    customer_staging=[]
    for i in range(1,161):
        c=customers[(i-1)%120]; email=c[3]
        if i>120: email='  '+email.upper()+'  '
        customer_staging.append((i,c[0],c[1],c[2],email,c[4],dt.datetime(2026,1,1)+dt.timedelta(days=i)))
    batch('customer_staging',['staging_id','source_customer_id','first_name','last_name','email','city','updated_at'],customer_staging)

    product_staging=[]
    for i in range(1,131):
        pr=products[(i-1)%80]
        sku=pr[1] if i<=80 else ' '+pr[1].lower()+' '
        product_staging.append((i,sku,pr[2],pr[3],money(pr[4]*(1+0.01*(i%4))),dt.datetime(2026,3,1)+dt.timedelta(hours=12*i)))
    batch('product_staging',['staging_id','sku','product_name','category','unit_price','updated_at'],product_staging)

    pipelines=['orders_bronze_to_silver','customer_daily','product_inventory','payments_ingest','events_stream']
    logs=[]
    for i in range(1,601):
        level='ERROR' if i%17==0 else ('WARN' if i%7==0 else 'INFO')
        msg={'ERROR':'Source connection timed out during ingestion','WARN':'Late-arriving records detected in batch','INFO':'Pipeline completed and data quality checks passed'}[level]
        logs.append((i,rng.choice(pipelines),level,msg,dt.datetime(2026,7,1,1)+dt.timedelta(hours=3*i)))
    for i in range(1,26): logs.append((600+i,*logs[i-1][1:]))
    batch('pipeline_logs',['log_id','pipeline_name','log_level','message','logged_at'],logs)

    with conn.cursor() as cur:
        cur.execute('CREATE INDEX idx_orders_customer_date ON public.orders(customer_id,order_date)')
        cur.execute('CREATE INDEX idx_events_user_ts ON public.events(user_id,event_ts)')
        cur.execute('CREATE INDEX idx_transactions_customer_ts ON public.transactions(customer_id,transaction_ts)')
        cur.execute('CREATE INDEX idx_order_items_order ON public.order_items(order_id)')
        cur.execute('CREATE INDEX idx_tickets_customer ON public.support_tickets(customer_id)')
        cur.execute('ANALYZE')
    conn.commit()
    print('\nSUCCESS: 15 tables seeded with realistic data.')
except Exception:
    conn.rollback()
    raise

## 3. Validate row counts and preview realistic names

In [ ]:
with conn.cursor() as cur:
    for table in TABLES:
        cur.execute(f'SELECT COUNT(*) FROM public.{table}')
        count=cur.fetchone()[0]
        assert count >= 50, (table,count)
        print(f'{table:22s} {count:>5}')
    cur.execute("SELECT COUNT(*) FROM public.customers WHERE first_name LIKE 'Customer%' OR last_name LIKE 'Lastname%'")
    assert cur.fetchone()[0] == 0
print('Validation passed.')

def preview(query):
    with conn.cursor() as cur:
        cur.execute(query)
        return pd.DataFrame(cur.fetchall(), columns=[x[0] for x in cur.description])

display(preview("SELECT customer_id, first_name, last_name, city, province FROM public.customers LIMIT 10"))
display(preview("SELECT sku, product_name, category, unit_price FROM public.products LIMIT 10"))

## Notes
- All data is **synthetic**, but names, products, prices, locations and business events are human-readable and internally consistent.
- Intentional edge cases remain: repeated logical events and logs, duplicate staging keys, payment retries, unresolved tickets, and 3 mismatched order totals.
- Counts may differ from the earlier seed (e.g., order items are now variable per order); the 80 questions depend on schema, not fixed row counts.
- **`public` is the target** so existing 80-question notebooks using `FROM orders` work without `SET search_path`.
- If you have custom tables with foreign keys referencing these 15 tables, the reset will fail safely rather than deleting those external tables.